# Significance gate on I(Y; V,S)

**What this tests.** Every headline result reports a percentage decomposition of
`I_joint`. Percentages of a near-zero, non-significant total are not merely
uninterpretable, they are numerically unstable — small absolute noise in the
atoms produces large swings in `*_pct` as the denominator shrinks. So before any
composition is quoted, `I_joint` itself has to clear its own permutation null.


In [ ]:
import sys
sys.path.insert(0, "../src")

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

import mimic_channels as mimic
import eicu_channels as eicu
import pid_broja as pb
import quantize as qz

pd.set_option("display.width", 160)

K = 13                      # fixed pre-collapse grid -- matches notebooks 02 and 03
# 300 s, not 90: a draw whose solver cascade times out costs up to 30 s, and a
# 90 s budget could be consumed by one or two of them (that is exactly what
# produced a p of 1/17 for MIMIC age_ge65 on the first k=13 run).
N_PERM, N_JOBS, BUDGET_S = 1000, 16, 300.0
GATE_ALPHA = 0.05
SEED = 0

MG_MIMIC = 50960            # magnesium: dropped from MIMIC to match eICU's panel, this is done to preserve the same laboratory panel in both dataset.

PANEL12_EICU = {"bicarbonate", "chloride", "creatinine", "glucose", "Hct", "Hgb",
                "platelets x 1000", "potassium", "RBC", "sodium", "BUN", "WBC x 1000"}
TARGETS = ["mortality", "gender_female", "age_ge65", "race_binary"]

pb._pool(N_JOBS)

cc_m = mimic.complete_case(itemid_scope=set(mimic.CORE) - {MG_MIMIC})
cc_e = eicu.complete_case(labname_scope=PANEL12_EICU)
DBS = [("MIMIC-IV", mimic, cc_m), ("eICU-CRD", eicu, cc_e)]
for db, _, cc in DBS:
    print(f"{db:9s} n={cc.meta['n']:7,d}  V={cc.Xv.shape[1]}d S={cc.Xs.shape[1]}d  "
          f"prev={cc.meta['prevalence']:.4f}")
print(f"\nk={K} fixed grid -> {K*K*2} joint cells, collapse_for_occupancy NOT applied")

MIMIC-IV  n= 64,623  V=24d S=24d  prev=0.1069
eICU-CRD  n=120,266  V=24d S=24d  prev=0.0947

k=13 fixed grid -> 338 joint cells, collapse_for_occupancy NOT applied


## 1. The gate

For each cell: discretise both channels on the fixed k=13 grid, decompose once to
get the observed `I_joint`, then shuffle the outcome against the fixed joint
cells and read the null off the summed atoms.

In [2]:
def ijoint_gate(ch, mod, k=K, seed=SEED):
    """Observed I_joint against its own permutation null.

    Fixed grid: kmeans_levels only, no occupancy collapse. One measure suffices
    -- the permuted count tables, and therefore I_joint's null, do not depend on
    the redundancy measure used to decompose them, only the atoms do."""
    v, _ = qz.kmeans_levels(qz.standardize_for_kmeans(ch.Xv), k, seed=seed)
    s, _ = qz.kmeans_levels(
        qz.standardize_for_kmeans(ch.Xs, sentinel=mod.TIME_SENTINEL), k, seed=seed)

    d = pb.decompose(v, s, ch.y, measure="broja")
    null, n_used = pb.permutation_null_budgeted(v, s, ch.y, n_perm=N_PERM,
                                                n_jobs=N_JOBS, budget_s=BUDGET_S,
                                                seed=seed, measure="broja")
    obs = d["mi_plugin_nats"]["I_joint"]
    null_ij = null["red"] + null["u_val"] + null["u_str"] + null["syn"]
    # A draw whose solver cascade failed comes back as NaN. It carries no
    # information about the null, and NaN compares False against everything, so
    # leaving it in would silently deflate the exceedance count AND poison the
    # null mean. Drop it and report how many were dropped.
    finite = np.isfinite(null_ij)
    n_drop = int((~finite).sum())
    null_ij = null_ij[finite]
    n_eff = int(null_ij.size)
    p = float(((null_ij >= obs).sum() + 1) / (n_eff + 1)) if n_eff else float("nan")
    return {
        "I_joint_obs": obs,
        "I_joint_null_mean": float(null_ij.mean()) if n_eff else float("nan"),
        "I_joint_debiased": obs - float(null_ij.mean()) if n_eff else float("nan"),
        "I_joint_p": p,
        "I_joint_gate_pass": bool(p <= GATE_ALPHA),
        "n_perm_used": n_used,
        "n_perm_effective": n_eff,
        "n_perm_dropped": n_drop,
        "min_cell": d["min_cell_count"],
        "n_empty_cells": d["n_empty_cells"],
        "converged": d["converged"],
    }

## 2. All eight cells

In [3]:
results_summary = []
demo_cache = {db: mod.demographic_targets()[0] for db, mod, _ in DBS}

for db, mod, cc in DBS:
    for target in TARGETS:
        ch = cc if target == "mortality" else mod.retarget(cc, demo_cache[db][target], target)
        t0 = time.time()
        g = ijoint_gate(ch, mod)
        name = "hospital_expire_flag" if target == "mortality" else target
        print(f"{db:9s} {name:20s} n={len(ch.y):7,d}  I={g['I_joint_obs']:.5f}  "
              f"null={g['I_joint_null_mean']:.5f}  ratio={g['I_joint_obs']/g['I_joint_null_mean']:5.0f}x  "
              f"p={g['I_joint_p']:.4f}  n={g['n_perm_effective']}(-{g['n_perm_dropped']})  "
              f"{'PASS' if g['I_joint_gate_pass'] else 'FAIL'}  "
              f"({time.time()-t0:.0f}s)", flush=True)
        results_summary.append({"dataset": db, "target": name, "k": K, "n": int(len(ch.y)), **g})

MIMIC-IV  hospital_expire_flag n= 64,623  I=0.03995  null=0.00132  ratio=   30x  p=0.0010  n=1000(-0)  PASS  (5s)


MIMIC-IV  gender_female        n= 64,623  I=0.02661  null=0.00131  ratio=   20x  p=0.0010  n=1000(-0)  PASS  (5s)


MIMIC-IV  age_ge65             n= 64,623  I=0.02498  null=0.00131  ratio=   19x  p=0.0069  n=143(-15)  PASS  (128s)


MIMIC-IV  race_binary          n= 48,361  I=0.01340  null=0.00177  ratio=    8x  p=0.0010  n=1000(-0)  PASS  (5s)


eICU-CRD  hospital_expire_flag n=120,266  I=0.01957  null=0.00071  ratio=   27x  p=0.0010  n=1000(-0)  PASS  (5s)


eICU-CRD  gender_female        n=120,224  I=0.02397  null=0.00070  ratio=   34x  p=0.0010  n=1000(-0)  PASS  (5s)


eICU-CRD  age_ge65             n=120,266  I=0.02597  null=0.00070  ratio=   37x  p=0.0010  n=1000(-0)  PASS  (5s)


eICU-CRD  race_binary          n=106,676  I=0.00742  null=0.00080  ratio=    9x  p=0.0010  n=1000(-0)  PASS  (5s)


## 3. Summary and save

In [4]:
summary_df = pd.DataFrame(results_summary)
summary_df["ratio_vs_null"] = summary_df.I_joint_obs / summary_df.I_joint_null_mean
print(summary_df[["dataset", "target", "n", "I_joint_obs", "I_joint_null_mean",
                  "ratio_vs_null", "I_joint_p", "I_joint_gate_pass"]]
      .to_string(index=False, float_format=lambda x: f"{x:.5f}"))

n_pass = int(summary_df.I_joint_gate_pass.sum())
print(f"\n{n_pass}/{len(summary_df)} cells clear the I_joint gate (p <= {GATE_ALPHA})")
print(f"ratio vs null spans {summary_df.ratio_vs_null.min():.0f}x to "
      f"{summary_df.ratio_vs_null.max():.0f}x")
print(f"BROJA converged on every cell: {bool(summary_df.converged.all())}")

payload = {
    "protocol": {"k": K, "grid": "fixed pre-collapse (collapse_for_occupancy NOT applied)",
                 "panel": "matched 12 analytes, 24-d channels on both databases",
                 "seed": SEED, "n_perm": N_PERM, "budget_s": BUDGET_S,
                 "measure": "broja",
                 "supersedes": ("the k=7 / k=6 occupancy-collapsed gate on each database's own "
                                "panel, which read headline_k from the retired "
                                "results/headline_result*.json files")},
    "cells": results_summary,
}
out = Path("../results/ijoint_gate_summary.json")
out.write_text(json.dumps(payload, indent=2, default=float))
print(f"wrote {out}")

 dataset               target      n  I_joint_obs  I_joint_null_mean  ratio_vs_null  I_joint_p  I_joint_gate_pass
MIMIC-IV hospital_expire_flag  64623      0.03995            0.00132       30.32417    0.00100               True
MIMIC-IV        gender_female  64623      0.02661            0.00131       20.26959    0.00100               True
MIMIC-IV             age_ge65  64623      0.02498            0.00131       19.03089    0.00694               True
MIMIC-IV          race_binary  48361      0.01340            0.00177        7.58015    0.00100               True
eICU-CRD hospital_expire_flag 120266      0.01957            0.00071       27.47819    0.00100               True
eICU-CRD        gender_female 120224      0.02397            0.00070       34.08805    0.00100               True
eICU-CRD             age_ge65 120266      0.02597            0.00070       37.03027    0.00100               True
eICU-CRD          race_binary 106676      0.00742            0.00080        9.31331    0